# Вероятность достижения ценового барьера по GARCH

Ноутбук строит прогноз дневной волатильности GARCH(1,1), моделирует будущие цены методом Монте-Карло и оценивает вероятность, что цена **на дату окончания срока** окажется не ниже барьера. В режиме GARCH условная дисперсия пересчитывается внутри каждой траектории после каждого смоделированного шока.

GARCH оценивается с t-распределением Стьюдента, и в Монте-Карло используются такие же стандартизированные t-шоки. Прогноз аналитиков на 12 месяцев задаёт ожидаемую доходность цены; он переводится в доходность одной свечи. Хвосты t-шоков усечены, после чего дрейф численно корректируется, чтобы средняя цена в модели соответствовала этому прогнозу.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from datetime import date, timedelta
from arch.univariate import ConstantMean, StudentsT
from arch.univariate.volatility import GARCH

# ===== ВХОДНЫЕ ПАРАМЕТРЫ: меняйте только эти значения =====
MONTHS = 4                 # срок прогноза в месяцах
BARRIER_PERCENT = 105.0    # 120 = цена на 20% выше начальной; 90 = на 10% ниже
# None: число прогнозных свечей в месяце выбирается по TIMEFRAME автоматически.
# Можно задать своё, например 42 для 4-часовых свечей.
FORECAST_PERIODS_PER_MONTH = None
N_SIMULATIONS = 100_000
RANDOM_SEED = 42
ANALYST_FORECAST_12M_PCT = 24  # ожидаемая доходность цены за 12 мес., например 15 = +15%
# None: прогноз волатильности будет построен GARCH ниже.
# Либо укажите число (например, 2.5 для постоянных 2.5% в день)
# или массив из horizon значений в процентах, полученный из другого GARCH-прогноза.
DAILY_VOL_FORECAST_PCT = None

# Данные Московской биржи (ISS API)
TICKER = 'LKOH'            # например: SBER, GAZP, LKOH, TATN
TIMEFRAME = 'day'          # day / week / month / 4h (также: день / неделя / месяц)
GARCH_OBSERVATIONS = 1000   # сколько последних свечей использовать для оценки GARCH
GARCH_MAX_PERSISTENCE = 0.995  # жёсткое ограничение: alpha + beta <= это значение
# =========================================================

if MONTHS <= 0:
    raise ValueError('Срок должен быть положительным.')
if BARRIER_PERCENT <= 0:
    raise ValueError('Барьер должен быть больше нуля.')
if ANALYST_FORECAST_12M_PCT <= -100:
    raise ValueError('ANALYST_FORECAST_12M_PCT должно быть больше -100%.')
if GARCH_OBSERVATIONS < 100:
    raise ValueError('Для GARCH укажите не менее 100 свечей.')
if not 0 < GARCH_MAX_PERSISTENCE < 1:
    raise ValueError('GARCH_MAX_PERSISTENCE должно быть между 0 и 1.')

barrier_multiplier = BARRIER_PERCENT / 100
print(f'Срок: {MONTHS} мес.')
print(f'Барьер: {BARRIER_PERCENT:.2f}% от стартовой цены')

ModuleNotFoundError: No module named 'numpy'

In [5]:
# 1. Получаем OHLCV-свечи напрямую из ISS API Московской биржи.
# ISS поддерживает 60 минут, день (24), неделю (7) и месяц (31).
# Для 4h загружаются часовые свечи и агрегируются в 4-часовые.
timeframe_aliases = {
    'day': 'day', 'день': 'day',
    'week': 'week', 'неделя': 'week',
    'month': 'month', 'месяц': 'month',
    '4h': '4h', '4ч': '4h', '4 часа': '4h',
}
timeframe = timeframe_aliases.get(str(TIMEFRAME).strip().lower())
if timeframe is None:
    raise ValueError("TIMEFRAME: 'day', 'week', 'month' или '4h'.")

# Число свечей на месяц должно соответствовать таймфрейму GARCH.
default_periods_per_month = {'day': 21, 'week': 52 / 12, 'month': 1, '4h': 63}
periods_per_month = (default_periods_per_month[timeframe]
                     if FORECAST_PERIODS_PER_MONTH is None
                     else FORECAST_PERIODS_PER_MONTH)
if periods_per_month <= 0:
    raise ValueError('FORECAST_PERIODS_PER_MONTH должно быть положительным.')
horizon = max(1, int(round(MONTHS * periods_per_month)))
print(f'Горизонт: {horizon} свечей {timeframe} ({MONTHS} мес.)')

iss_interval = {'day': 24, 'week': 7, 'month': 31, '4h': 60}[timeframe]
lookback_days = {
    'day': max(GARCH_OBSERVATIONS * 3, 365),
    'week': max(GARCH_OBSERVATIONS * 10, 730),
    'month': max(GARCH_OBSERVATIONS * 45, 3650),
    '4h': max(GARCH_OBSERVATIONS * 2, 180),
}[timeframe]

import time

# Сессия с «браузерным» User-Agent: ISS MOEX сбрасывает соединения
# для запросов с дефолтным python-requests UA.
_MOEX = requests.Session()
_MOEX.headers.update({
    'User-Agent': ('Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                   'AppleWebKit/537.36 (KHTML, like Gecko) '
                   'Chrome/124.0 Safari/537.36'),
    'Accept': 'application/json, text/plain, */*',
    'Referer': 'https://iss.moex.com/',
})

def _moex_get(url, params, retries=4):
    last = None
    for attempt in range(retries):
        try:
            r = _MOEX.get(url, params=params, timeout=30)
            r.raise_for_status()
            return r
        except requests.RequestException as exc:
            last = exc
            time.sleep(1.5 * (attempt + 1))
    raise ConnectionError(
        f'ISS MOEX недоступен ({last}). Проверьте сеть/VPN с российским IP.'
    )

def load_moex_candles(ticker, interval, date_from, date_till):
    url = (f'https://iss.moex.com/iss/engines/stock/markets/shares/'
           f'securities/{ticker.upper()}/candles.json')
    chunks, start = [], 0
    while True:
        response = _moex_get(url, {
            'from': date_from.isoformat(), 'till': date_till.isoformat(),
            'interval': interval, 'start': start, 'iss.meta': 'off'
        })
        payload = response.json()['candles']
        batch = pd.DataFrame(payload['data'], columns=payload['columns'])
        if batch.empty:
            break
        chunks.append(batch)
        if len(batch) < 500:
            break
        start += len(batch)
    if not chunks:
        raise ValueError(f'ISS не вернул свечи для {ticker}. Проверьте тикер и период.')
    return pd.concat(chunks, ignore_index=True)

raw_candles = load_moex_candles(
    TICKER, iss_interval, date.today() - timedelta(days=lookback_days), date.today()
)
raw_candles['begin'] = pd.to_datetime(raw_candles['begin'])
raw_candles = raw_candles.sort_values('begin').drop_duplicates('begin').set_index('begin')

if timeframe == '4h':
    candles = raw_candles.resample('4h', origin='start_day', offset='2h').agg({
        'open': 'first', 'high': 'max', 'low': 'min', 'close': 'last',
        'value': 'sum', 'volume': 'sum'
    }).dropna(subset=['close'])
else:
    candles = raw_candles

candles = candles.tail(GARCH_OBSERVATIONS)
if len(candles) < GARCH_OBSERVATIONS:
    raise ValueError(
        f'Для {TICKER} доступно только {len(candles)} свечей выбранного периода; '
        'уменьшите GARCH_OBSERVATIONS или смените период.'
    )
prices = candles['close'].astype(float).reset_index(drop=True)
print(f'Загружено {len(candles)} свечей {TICKER} ({timeframe}); '
      f'{candles.index[0]} — {candles.index[-1]}')
returns_pct = (100 * np.log(prices / prices.shift(1))).dropna()

class CappedGARCH(GARCH):
    """GARCH с ограничением alpha + beta <= max_persistence."""

    def __init__(self, *args, max_persistence, **kwargs):
        super().__init__(*args, **kwargs)
        self.max_persistence = max_persistence

    def constraints(self):
        a, b = super().constraints()
        # Последняя строка базового ограничения: -(alpha + beta) >= -1.
        # Заменяем 1 на выбранный верхний предел, например 0.995.
        b[-1] = -self.max_persistence
        return a, b

if DAILY_VOL_FORECAST_PCT is None:
    garch = ConstantMean(
        returns_pct,
        volatility=CappedGARCH(p=1, q=1, max_persistence=GARCH_MAX_PERSISTENCE),
        distribution=StudentsT(),
    )
    garch_result = garch.fit(disp='off')
    forecast = garch_result.forecast(horizon=horizon, reindex=False)
    # arch выдаёт дисперсию в процентах²; переводим в доли, например 2% -> 0.02
    daily_vol = np.sqrt(forecast.variance.iloc[-1].to_numpy()) / 100
    omega = garch_result.params['omega'] / 100**2  # перевод из процентов² в доли²
    alpha = garch_result.params['alpha[1]']
    beta = garch_result.params['beta[1]']
    persistence = alpha + beta
    nu = garch_result.params['nu']
    print(f'alpha={alpha:.6f}, beta={beta:.6f}, alpha + beta={persistence:.6f} '
          f'(лимит {GARCH_MAX_PERSISTENCE:.3f})')
else:
    supplied_vol = np.asarray(DAILY_VOL_FORECAST_PCT, dtype=float)
    if supplied_vol.ndim == 0:
        supplied_vol = np.full(horizon, supplied_vol)
    if supplied_vol.size != horizon or np.any(supplied_vol < 0):
        raise ValueError('Нужна положительная волатильность на каждый из horizon торговых дней.')
    daily_vol = supplied_vol / 100
    omega = alpha = beta = None
    # Если волатильность передана вручную, задаём nu отдельно.
    nu = 4.0

start_price = float(prices.iloc[-1])
barrier_price = start_price * barrier_multiplier
periods_per_year = 12 * periods_per_month
target_log_gross_per_period = np.log1p(ANALYST_FORECAST_12M_PCT / 100) / periods_per_year

print(f'Последняя цена: {start_price:.2f}')
print(f'Цена-барьер: {barrier_price:.2f}')
print(f'Прогноз дневной волатильности: {daily_vol[0]:.2%} → {daily_vol[-1]:.2%}')
print(f'Степени свободы t-распределения (nu): {nu:.2f}')
print(f'Прогноз аналитиков на 12 мес.: {ANALYST_FORECAST_12M_PCT:.2f}% '      f'(целевой лог-рост на свечу: {target_log_gross_per_period:.6%})')

NameError: name 'TIMEFRAME' is not defined

In [6]:
# 2. Path-dependent Монте-Карло с теми же t-инновациями, что использовались при оценке GARCH.
# Умножение на sqrt((nu - 2) / nu) нормирует дисперсию t-шока к единице.
# Ограничение хвостов нужно для exp(log_return): у чистого t-распределения
# нет конечного экспоненциального момента, поэтому редкие шоки могут давать нереальные цены.
T_SHOCK_CLIP_QUANTILE = 0.995
rng = np.random.default_rng(RANDOM_SEED)
if nu <= 2:
    raise ValueError('Для стандартизации t-шоков требуется nu > 2.')
if not 0.5 < T_SHOCK_CLIP_QUANTILE < 1:
    raise ValueError('T_SHOCK_CLIP_QUANTILE должен быть между 0.5 и 1.')

t_clip = (
    np.quantile(rng.standard_t(df=nu, size=2_000_000),
                [1 - T_SHOCK_CLIP_QUANTILE, T_SHOCK_CLIP_QUANTILE])
    * np.sqrt((nu - 2) / nu)
 )
rng = np.random.default_rng(RANDOM_SEED)

# Коррекция дрейфа: E[exp(sqrt(h) * Z)] считается для усечённого t-шока.
# Так средняя цена следует прогнозу аналитиков, а не только медианная траектория.
correction_rng = np.random.default_rng(RANDOM_SEED + 1)
correction_shocks = correction_rng.standard_t(df=nu, size=25_000)
correction_shocks = np.clip(
    correction_shocks * np.sqrt((nu - 2) / nu), t_clip[0], t_clip[1]
)
variance_grid_max = max(1.0, 1_000 * float(np.max(daily_vol**2)))
variance_grid = np.r_[0.0, np.geomspace(1e-12, variance_grid_max, 500)]
log_exp_moment_grid = np.array([
    np.log(np.mean(np.exp(np.sqrt(h) * correction_shocks)))
    for h in variance_grid
])

def analyst_drift(variance):
    """Дрейф лог-доходности, дающий прогноз аналитиков в среднем."""
    if np.max(variance) > variance_grid_max:
        raise RuntimeError('Слишком большая GARCH-дисперсия для drift correction.')
    convexity_correction = np.interp(variance, variance_grid, log_exp_moment_grid)
    return target_log_gross_per_period - convexity_correction

simulated_prices = np.empty((N_SIMULATIONS, horizon))
current_prices = np.full(N_SIMULATIONS, start_price)

if DAILY_VOL_FORECAST_PCT is None:
    # h_1 — прогноз дисперсии на следующий торговый день.
    # Далее каждая траектория обновляет h: omega + alpha * epsilon² + beta * h.
    current_variance = np.full(N_SIMULATIONS, daily_vol[0]**2)
    for day in range(horizon):
        shock = rng.standard_t(df=nu, size=N_SIMULATIONS)
        shock = np.clip(shock * np.sqrt((nu - 2) / nu), t_clip[0], t_clip[1])
        log_return = analyst_drift(current_variance) + np.sqrt(current_variance) * shock
        current_prices *= np.exp(log_return)
        simulated_prices[:, day] = current_prices
        innovation_sq = current_variance * shock**2
        current_variance = omega + alpha * innovation_sq + beta * current_variance
    print('Режим симуляции: path-dependent GARCH(1,1) с t-шоками')
else:
    # При ручном вводе нет omega/alpha/beta, поэтому используется заданная детерминированная волатильность.
    for day, sigma in enumerate(daily_vol):
        shock = rng.standard_t(df=nu, size=N_SIMULATIONS)
        shock = np.clip(shock * np.sqrt((nu - 2) / nu), t_clip[0], t_clip[1])
        current_prices *= np.exp(analyst_drift(sigma**2) + sigma * shock)
        simulated_prices[:, day] = current_prices
    print('Режим симуляции: вручную заданная волатильность и t-шоки')

terminal_prices = simulated_prices[:, -1]

probability_above = np.mean(terminal_prices >= barrier_price)
probability_below = 1 - probability_above
median_price = np.median(terminal_prices)
ci_5, ci_95 = np.quantile(terminal_prices, [0.05, 0.95])

print(f'Ограничение t-шоков: [{t_clip[0]:.2f}; {t_clip[1]:.2f}]')
print(f'Прогноз аналитиков используется как ожидаемая доходность цены: {ANALYST_FORECAST_12M_PCT:.2f}% за 12 мес.')
print(f'Вероятность цены ≥ {barrier_price:.2f} ({BARRIER_PERCENT:.1f}%): {probability_above:.2%}')
print(f'Вероятность цены < барьера: {probability_below:.2%}')
print(f'Медиана цены через {MONTHS} мес.: {median_price:.2f}')
print(f'90% интервал результатов: [{ci_5:.2f}; {ci_95:.2f}]')

NameError: name 'np' is not defined

In [4]:
# 3. Визуализация распределения конечной цены и части траекторий
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

n_paths_to_show = min(300, N_SIMULATIONS)
axes[0].plot(simulated_prices[:n_paths_to_show].T, alpha=0.08, color='steelblue')
axes[0].axhline(barrier_price, color='crimson', ls='--', label=f'Барьер: {barrier_price:.2f}')
axes[0].set(title='Симулированные траектории цены', xlabel='Торговый день', ylabel='Цена')
axes[0].legend()
axes[0].grid(alpha=0.25)

# Показываем центральные 99% результатов, чтобы редкие хвосты не сжимали гистограмму.
display_low, display_high = np.quantile(terminal_prices, [0.005, 0.995])
axes[1].hist(
    terminal_prices, bins=80, range=(display_low, display_high),
    density=True, color='steelblue', alpha=0.75
 )
axes[1].axvline(barrier_price, color='crimson', ls='--', label=f'Барьер: {barrier_price:.2f}')
axes[1].axvline(median_price, color='darkgreen', ls=':', label=f'Медиана: {median_price:.2f}')
axes[1].set_xlim(display_low, display_high)
axes[1].set(title='Распределение цены в конце срока (99% результатов)', xlabel='Цена', ylabel='Плотность')
axes[1].legend()
axes[1].grid(alpha=0.25)
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined